# Approach 1: Classical computer vision + SVM

Use normalized HSV and CIELAB color histograms plus a Local Binary Pattern texture histogram. The SVM is class-balanced. Tune choices using validation only; the supplied test set remains untouched until notebook 4.

In [1]:
from pathlib import Path
import sys

import joblib
from sklearn.metrics import accuracy_score, classification_report

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset import get_dataset_splits
from src.classical_model import build_classical_classifier, extract_feature_matrix

splits = get_dataset_splits(PROJECT_ROOT / 'data' / 'raw', validation_size=0.15, random_state=42)
class_names = splits['class_names']
print({name: len(splits[name]) for name in ('train', 'validation', 'test')})
print('Class order:', class_names)

{'train': 842, 'validation': 149, 'test': 99}
Class order: ['blackspot', 'canker', 'fresh', 'grenning']


In [2]:
X_train, y_train = extract_feature_matrix(splits['train'])
X_validation, y_validation = extract_feature_matrix(splits['validation'])
print('Feature matrix:', X_train.shape)
classifier = build_classical_classifier()
classifier.fit(X_train, y_train)
validation_predictions = classifier.predict(X_validation)
print('Validation accuracy:', round(accuracy_score(y_validation, validation_predictions), 4))
print(classification_report(y_validation, validation_predictions, labels=class_names, zero_division=0))

Feature matrix: (842, 91)
Validation accuracy: 0.9799
              precision    recall  f1-score   support

   blackspot       0.93      1.00      0.97        28
      canker       1.00      0.89      0.94        27
       fresh       0.98      1.00      0.99        42
    grenning       1.00      1.00      1.00        52

    accuracy                           0.98       149
   macro avg       0.98      0.97      0.97       149
weighted avg       0.98      0.98      0.98       149



In [3]:
# Once the validation result is acceptable, refit on all supplied training data.
X_all_train, y_all_train = extract_feature_matrix(splits['train'] + splits['validation'])
final_classical_model = build_classical_classifier()
final_classical_model.fit(X_all_train, y_all_train)
model_dir = PROJECT_ROOT / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
classical_model_path = model_dir / 'orange_classical.joblib'
joblib.dump(final_classical_model, classical_model_path)
print('Saved:', classical_model_path)

Saved: c:\green pods assignment2\avocado_ripening_classification\models\orange_classical.joblib
